In [1]:
import os

In [2]:
pwd = os.getcwd()

In [3]:
os.chdir("../")
print(os.getcwd())

/Users/xenan.bilgin/Projects/MLops/mlops-project1-wine-quality


In [4]:
import pandas as pd

data = pd.read_csv("artifacts/data_ingestion/winequality-red.csv")

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


In [6]:
# Create schema.yaml file for data validation
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class ModelTrainingConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [6]:
from src.mlops1_data_science_project import logger
from src.mlops1_data_science_project.constants import (
    CONFIG_FILE_PATH,
    PARAMS_FILE_PATH,
    SCHEMA_FILE_PATH,
)
from src.mlops1_data_science_project.utils.common import create_directories, read_yaml

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_training_config(self) -> ModelTrainingConfig:
        config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.target_column
        create_directories([config.root_dir])

        model_training_config = ModelTrainingConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema,
        )
        return model_training_config

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.linear_model import ElasticNet


class ModelTrainer:
    def __init__(self, config: ModelTrainingConfig):
        self.config = config

    def train(self):
        # Load training data
        train_data = pd.read_csv(self.config.train_data_path)
        x_train = train_data.drop(columns=[self.config.target_column])
        y_train = train_data[self.config.target_column]

        # Train model (example with ElasticNet)
        model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio)
        model.fit(x_train, y_train)

        # Save the trained model
        model_save_path = Path(self.config.root_dir) / f"{self.config.model_name}.pkl"

        joblib.dump(model, model_save_path)

In [ ]:
try:
    config_manager = ConfigurationManager()
    model_training_config = config_manager.get_model_training_config()
    model_trainer = ModelTrainer(config=model_training_config)
    model_trainer.train()
except Exception as e:
    logger.exception(f"Error occurred during model training: {e}")

[2026-05-27 16:49:37,994: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-27 16:49:37,995: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-27 16:49:37,997: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-05-27 16:49:37,998: INFO: common: created directory at: artifacts]
[2026-05-27 16:49:37,998: INFO: common: created directory at: artifacts/data_validation]
[2026-05-27 16:49:38,000: INFO: 1253244569: All columns are valid]
[2026-05-27 16:49:38,001: INFO: 2729420827: Data validation status: True]
